# Notebook: Pipeline de sklearn reproducible

Este notebook construye un pipeline ***100% reproducible y apto para producción*** para un dataset tabular siguiendo las transformaciones de feature engineering previamente definidas.

---

## 1. Imports obligatorios

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, Binarizer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

from feature_engine.imputation import CategoricalImputer, MeanMedianImputer, AddMissingIndicator
from feature_engine.encoding import RareLabelEncoder, OrdinalEncoder
from feature_engine.transformation import LogTransformer, YeoJohnsonTransformer
from feature_engine.wrappers import SklearnTransformerWrapper
from feature_engine.selection import DropFeatures

---

## 2. Carga de datos y split

In [ ]:
data = pd.read_csv('data/train.csv')

# Suponiendo que la variable target es la última columna y se llama 'SalePrice'; adaptar si cambia el dataset.
TARGET = 'SalePrice'
X = data.drop(TARGET, axis=1)
y = data[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=0
)

---

## 3. Transformación de y si es continua

In [ ]:
# Si el target es continuo (e.g. regresión), transformamos con np.log.
# Si fuera clasificación, omítase esta celda.
y_train = np.log(y_train)
y_test = np.log(y_test)

---

## 4. Definición de CONSTANTES de variables

> Adapta estos nombres y listas conforme a la ingeniería previa sobre tu dataset.

In [ ]:
# Lista ficticia de variables para ilustrar la estructura. Sustituye por tus propias variables reales.
CATEGORICAL_VARS_WITH_NA_MISSING = ["Alley", "PoolQC", "Fence"]
CATEGORICAL_VARS_WITH_NA_FREQUENT = ["MSZoning", "Functional", "KitchenQual", "Exterior1st", "Exterior2nd"]
NUMERICAL_VARS_WITH_NA = ["LotFrontage", "MasVnrArea", "GarageYrBlt"]
TEMPORAL_VARS = ["YearBuilt", "YearRemodAdd", "GarageYrBlt"]
REF_VAR = "YrSold"
NUMERICALS_LOG_VARS = ["GrLivArea", "LotArea", "MasVnrArea"]
NUMERICALS_YEO_VARS = ["1stFlrSF", "BsmtFinSF1"]
BINARIZE_VARS = ["ScreenPorch"]

# Ordinales de calidad
QUAL_VARS = ["ExterQual", "ExterCond", "BsmtQual", "BsmtCond", "HeatingQC", "KitchenQual", "FireplaceQu", "GarageQual", "GarageCond"]
QUAL_MAPPINGS = {
    "Po": 1,  # Poor
    "Fa": 2,  # Fair
    "TA": 3,  # Typical/Average
    "Gd": 4,  # Good
    "Ex": 5   # Excellent
}

# Variables nominales para rare label encoding y ordinal encoding
CATEGORICAL_VARS = [
    'MasVnrType', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
    'Electrical', 'MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig',
    'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle',
    'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'Foundation', 'Heating', 'CentralAir',
    'Functional', 'GarageType', 'GarageFinish', 'PavedDrive', 'MiscFeature', 'SaleType', 'SaleCondition'
]

---

## 5. Definición de CUSTOM TRANSFORMERS

In [ ]:
class TemporalVariableTransformer(BaseEstimator, TransformerMixin):
    """
    Calcula la edad de la variable temporal respecto a una referencia.
    """
    def __init__(self, variables, reference_variable):
        self.variables = variables
        self.reference_variable = reference_variable
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        for var in self.variables:
            X[var] = X[self.reference_variable] - X[var]
        return X

class Mapper(BaseEstimator, TransformerMixin):
    """
    Aplica mappings a variables ordinales de calidad, exposición, etc.
    """
    def __init__(self, variables, mappings):
        self.variables = variables
        self.mappings = mappings
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        for var in self.variables:
            X[var] = X[var].map(self.mappings)
        return X

---

## 6. Construcción del PIPELINE (ajusta las listas de variables según tu análisis)

In [ ]:
pipeline = Pipeline([
    # 1. IMPUTATION
    ("cat_imputer_missing", CategoricalImputer(
        imputation_method='missing', variables=CATEGORICAL_VARS_WITH_NA_MISSING)),
    ("cat_imputer_frequent", CategoricalImputer(
        imputation_method='frequent', variables=CATEGORICAL_VARS_WITH_NA_FREQUENT)),
    ("add_missing_ind", AddMissingIndicator(
        variables=NUMERICAL_VARS_WITH_NA)),
    ("num_imputer_mean", MeanMedianImputer(
        imputation_method='mean', variables=NUMERICAL_VARS_WITH_NA)),

    # 2. TEMPORAL
    ("temporal_transformer", TemporalVariableTransformer(
        variables=TEMPORAL_VARS, reference_variable=REF_VAR)),
    ("drop_ref_var", DropFeatures(features_to_drop=[REF_VAR])),

    # 3. TRANSFORMATIONS
    ("log_transformer", LogTransformer(variables=NUMERICALS_LOG_VARS)),
    ("yeo_transformer", YeoJohnsonTransformer(variables=NUMERICALS_YEO_VARS)),
    ("binarizer", SklearnTransformerWrapper(
        transformer=Binarizer(threshold=0), variables=BINARIZE_VARS)),

    # 4. MAPPERS (ordinales)
    ("qual_mapper", Mapper(
        variables=QUAL_VARS, mappings=QUAL_MAPPINGS)),

    # 5. ENCODING NOMINAL
    ("rare_label_enc", RareLabelEncoder(
        tol=0.01, n_categories=1, variables=CATEGORICAL_VARS)),
    ("ordinal_enc", OrdinalEncoder(
        encoding_method="ordered", variables=CATEGORICAL_VARS)),
])

---

## 7. Entrenamiento del Pipeline SOLO con train

In [ ]:
pipeline.fit(X_train, y_train)

---

## 8. Transformación de los datos

In [ ]:
X_train_transformed = pipeline.transform(X_train)
X_test_transformed = pipeline.transform(X_test)

---

## 9. Chequeo de NaN

In [ ]:
print("¿NaN en X_train_transformed?", np.any(pd.DataFrame(X_train_transformed).isnull().any()))
print("¿NaN en X_test_transformed?", np.any(pd.DataFrame(X_test_transformed).isnull().any()))

---

## 10. Serialización del pipeline

In [ ]:
joblib.dump(pipeline, 'proyecto_pipeline_machine_learning_muestra/preprocessors.pkl')